# 🌐 LIVE DEPLOYMENT URL: **PENDING** (replace this text with the real link after the app is deployed)

**Live app:** `https://YOUR-APP-NAME.streamlit.app`  ← *not deployed yet; do not submit with this placeholder*

---
# Real Estate Price Prediction Using Machine Learning and 10-Fold Cross-Validation
**University:** COMSATS University Islamabad, Lahore Campus  
**Course:** Machine Learning Fundamentals (AIC354)  
**Assignment 3:** Applying, Evaluating, and Comparing Multiple ML Algorithms on a Large Real-World Dataset

## 1. Project Introduction
House prices depend on many things: how old the house is, how close it is to public transport, how many shops are nearby, and where it is located.
In this project I use machine learning to learn the relationship between these factors and the price of a house.
I compare **five regression algorithms** using **10-Fold Cross-Validation** and measure them with **RMSE**.
The best model is saved and used in a web application deployed on the cloud.

## 2. Problem Statement
Given the transaction date, house age, distance to the nearest MRT station, number of convenience stores, latitude and longitude of a property,
predict the **house price of unit area** (a number). This is a **supervised regression** problem, because the dataset has known answers (labels) and the answer is a continuous number.

## 3. Objectives
1. Load and understand the real estate dataset.
2. Evaluate five regression algorithms (different from the Car Price example) with 10-Fold Cross-Validation.
3. Compare them using RMSE (primary metric), plus MAE and R² (supplementary).
4. Select the model with the lowest mean RMSE and train it on the full dataset.
5. Save the model and use it in a Streamlit web application.

## 4. Dataset Source and Description
* **Source:** Kaggle – Real Estate Price Prediction (quantbruce): https://www.kaggle.com/datasets/quantbruce/real-estate-price-prediction  
* **File:** `Real estate.csv` (commonly 414 rows). The notebook **checks the real row count** instead of assuming it.
* **Target:** `Y house price of unit area`
* **Features:** `X1 transaction date`, `X2 house age`, `X3 distance to the nearest MRT station`, `X4 number of convenience stores`, `X5 latitude`, `X6 longitude`
* `No` is only a row ID, so it is **not** used as an input.

**How to get the file:** download it from the Kaggle link above (free account needed), unzip it, and upload `Real estate.csv` when the notebook asks in Section 6.

## 5. Importing Libraries

In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# The five algorithms for this assignment (none of them is used in the Car Price example)
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

print("scikit-learn version:", sklearn.__version__)
print("pandas version      :", pd.__version__)

## 6. Loading the Dataset
Run the next cell and use the **Choose Files** button to upload `Real estate.csv` from your computer.
(If the file already exists in the Colab folder, the upload step is skipped.)

In [ ]:
CSV_NAME = "Real estate.csv"

if not os.path.exists(CSV_NAME):
    try:
        from google.colab import files          # only exists inside Google Colab
        uploaded = files.upload()               # opens the upload button
        CSV_NAME = list(uploaded.keys())[0]     # use whatever name was uploaded
    except ImportError:
        raise FileNotFoundError("Put 'Real estate.csv' in the same folder as this notebook.")

df_original = pd.read_csv(CSV_NAME)             # original data (kept untouched)
df = df_original.copy()                         # working copy
df.columns = df.columns.str.strip()             # remove accidental spaces in column names
print("Loaded file:", CSV_NAME)
df.head()                                       # first five rows

## 7. Data Exploration and Visualization

In [ ]:
print("Rows, columns:", df.shape)
print("\nColumn names and data types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

In [ ]:
# Define the target and the input features (names must match the dataset exactly)
TARGET = "Y house price of unit area"
FEATURES = ["X1 transaction date",
            "X2 house age",
            "X3 distance to the nearest MRT station",
            "X4 number of convenience stores",
            "X5 latitude",
            "X6 longitude"]

missing_cols = [c for c in FEATURES + [TARGET] if c not in df.columns]
assert not missing_cols, f"These columns were not found in the file: {missing_cols}. Actual columns: {list(df.columns)}"
print("Target  :", TARGET)
print("Features:", FEATURES)
print("Ignored :", [c for c in df.columns if c not in FEATURES + [TARGET]], "(record identifier)")

In [ ]:
# Visualization 1: distribution of the target, and each feature vs price
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()
axes[0].hist(df[TARGET], bins=25, edgecolor="black")
axes[0].set_title("Target: house price of unit area")
for ax, col in zip(axes[1:], FEATURES):
    ax.scatter(df[col], df[TARGET], s=10, alpha=0.6)
    ax.set_xlabel(col, fontsize=8); ax.set_ylabel("price")
    ax.set_title(col.split(" ", 1)[1], fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Visualization 2: correlation between features and the target
corr = df[FEATURES + [TARGET]].corr()
plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="correlation")
labels = [c.split(" ")[0] for c in corr.columns[:-1]] + ["Y"]
plt.xticks(range(len(labels)), labels); plt.yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation matrix"); plt.show()

## 8. Data Cleaning and Preprocessing
* The record identifier `No` is excluded from the inputs.
* Missing values are handled **only if the check above found any** (the next cell stops with a message if there are, so the choice is made consciously).
* Scaling (needed by SVR, MLP and Gaussian Process) is done **inside the pipelines**, so it is learned only from the training folds. This prevents *data leakage*.

In [ ]:
X = df[FEATURES]          # feature table (order matters and is reused in the web app)
y = df[TARGET]            # target values

print("Missing values in X:", int(X.isnull().sum().sum()), "| in y:", int(y.isnull().sum()))
if X.isnull().sum().sum() > 0 or y.isnull().sum() > 0:
    raise ValueError("Missing values found. Decide how to handle them (e.g. SimpleImputer in the pipeline) before continuing.")
print("All feature types numeric:", all(pd.api.types.is_numeric_dtype(X[c]) for c in FEATURES))
print("Target range:", y.min(), "to", y.max())
print("X shape:", X.shape, "| y shape:", y.shape)

## 9. The Five Regression Algorithms (in simple words)
| # | Algorithm | Simple idea | Why it suits numerical prediction |
|---|-----------|-------------|-----------------------------------|
| 1 | **Support Vector Regression (SVR)** | Fits a smooth curve that tries to stay within a small "tube" around the data points and ignores small errors inside the tube. With the RBF kernel it can bend to follow non-linear patterns. | Predicts a number directly and works well on small/medium datasets, but needs scaled features. |
| 2 | **AdaBoost Regressor** | Builds many very small decision trees one after another. Each new tree pays more attention to the houses the earlier trees predicted badly. | Combines many weak learners into a strong numeric predictor. |
| 3 | **Extra Trees Regressor** | Builds many decision trees with extra randomness in how they split, then averages their predictions. | Averaging reduces variance, handles non-linear relationships and needs little preprocessing. |
| 4 | **MLP Regressor (neural network)** | Layers of connected "neurons" learn a non-linear formula from inputs to price by adjusting weights step by step. | Can learn complex relations; outputs a continuous number. Needs scaled features. |
| 5 | **Gaussian Process Regressor** | Treats price as a smooth function: houses with similar features should have similar prices. It uses similarity (a kernel) between houses to predict. | Works well on small datasets like this one and learns how important each feature is. |

*Note:* AdaBoost internally uses small decision trees as its building blocks (that is how boosting works), but the model being evaluated is AdaBoost itself, which is not one of the instructor's five.  
**Not used (instructor's five):** Linear Regression, KNN, Decision Tree, Random Forest, Gradient Boosting.

In [ ]:
# Pipelines: scaling is inside, so each fold learns its own scaling from its training part only.
# For SVR and MLP, the target is also standardised (TransformedTargetRegressor) so the model trains
# more reliably when prices are around tens of units. Predictions are converted back automatically.
RANDOM_STATE = 42

models = {
    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", TransformedTargetRegressor(
            regressor=SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale"),
            transformer=StandardScaler())),
    ]),
    "AdaBoost": Pipeline([
        ("model", AdaBoostRegressor(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE)),
    ]),
    "Extra Trees": Pipeline([
        ("model", ExtraTreesRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "MLP Regressor": Pipeline([
        ("scaler", StandardScaler()),
        ("model", TransformedTargetRegressor(
            regressor=MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu",
                                   max_iter=3000, random_state=RANDOM_STATE),
            transformer=StandardScaler())),
    ]),
    "Gaussian Process": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianProcessRegressor(
            kernel=C(1.0) * RBF(length_scale=np.ones(X.shape[1])) + WhiteKernel(noise_level=0.1),
            normalize_y=True, n_restarts_optimizer=2, random_state=RANDOM_STATE)),
    ]),
}
print("Models ready:", list(models.keys()))

## 10. 10-Fold Cross-Validation
The data is split into 10 equal parts (folds). For each round, the model trains on 9 folds and is tested on the 1 remaining fold, so **every row is used for testing exactly once**.
All five models use the **same 10 folds** (`KFold(n_splits=10, shuffle=True, random_state=42)`).

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
print("Number of folds:", kf.get_n_splits(X))
for i, (tr, te) in enumerate(kf.split(X), 1):
    print(f"Fold {i:2d}: train = {len(tr)} rows, test = {len(te)} rows")

## 11. Model Evaluation
The function below trains each model 10 times (once per fold), records RMSE, MAE and R² on the **test fold**, and prints real progress with elapsed and estimated remaining time.
Any warning (for example a convergence warning) is counted and shown, not hidden.

In [ ]:
def evaluate_model(name, pipeline):
    fold_rmse, fold_mae, fold_r2 = [], [], []
    start = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for i, (train_idx, test_idx) in enumerate(kf.split(X), 1):
            model = clone(pipeline)                                  # fresh, unfitted copy every fold
            model.fit(X.iloc[train_idx], y.iloc[train_idx])          # train on 9 folds
            pred = model.predict(X.iloc[test_idx])                   # test on the 1 held-out fold
            fold_rmse.append(np.sqrt(mean_squared_error(y.iloc[test_idx], pred)))
            fold_mae.append(mean_absolute_error(y.iloc[test_idx], pred))
            fold_r2.append(r2_score(y.iloc[test_idx], pred))
            elapsed = time.time() - start
            remaining = elapsed / i * (10 - i)
            print(f"  [{name}] fold {i:2d}/10 | RMSE = {fold_rmse[-1]:7.3f} | elapsed {elapsed:6.1f}s | est. remaining {remaining:6.1f}s")
    total = time.time() - start
    warn_msgs = sorted({str(w.category.__name__) for w in caught})
    if warn_msgs:
        print(f"  [{name}] warnings raised: {warn_msgs} (count={len(caught)})")
    return {"Model": name, "Mean RMSE": np.mean(fold_rmse), "Std RMSE": np.std(fold_rmse, ddof=1),
            "Mean MAE": np.mean(fold_mae), "Mean R2": np.mean(fold_r2),
            "Time (s)": total, "Warnings": len(caught),
            **{f"Fold {k+1} RMSE": v for k, v in enumerate(fold_rmse)}}

results = []
for name, pipe in models.items():
    print(f"\n=== Evaluating {name} ===")
    results.append(evaluate_model(name, pipe))
print("\nAll five models finished all 10 folds.")

> **If a model shows warnings** (for example `ConvergenceWarning` for the MLP), look at the count above. If it happens, increase `max_iter` for that model in Section 9 and re-run, and write the change in your report. Do not hide it. A `ConvergenceWarning` from the Gaussian Process usually means one kernel setting reached its allowed limit; it is not an error, but mention it in your report if it appears.

## 12. Comparison Table (sorted by mean RMSE, lowest = best)

In [ ]:
results_df = pd.DataFrame(results).sort_values("Mean RMSE").reset_index(drop=True)
results_df.index = results_df.index + 1          # rank starts at 1
summary_cols = ["Model", "Mean RMSE", "Std RMSE", "Mean MAE", "Mean R2", "Time (s)", "Warnings"]
results_df[summary_cols].round(4)

In [ ]:
# Per-fold RMSE values for all models
results_df[["Model"] + [f"Fold {k} RMSE" for k in range(1, 11)]].round(3)

## 13. RMSE Comparison Chart

In [ ]:
plt.figure(figsize=(9, 5))
bars = plt.bar(results_df["Model"], results_df["Mean RMSE"], yerr=results_df["Std RMSE"],
               capsize=5, color="steelblue", edgecolor="black")
for b, v in zip(bars, results_df["Mean RMSE"]):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.05, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.xlabel("Algorithm")
plt.ylabel("Mean RMSE over 10 folds (lower is better)")
plt.title("10-Fold CV RMSE Comparison - Real Estate Price Prediction\n(error bars = standard deviation across folds)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("rmse_comparison.png", dpi=150)
plt.show()

## 14. Selection of the Best Model
The best model is the one with the **lowest mean cross-validation RMSE**, decided only from the measured results above.

In [ ]:
best_name = results_df.loc[1, "Model"]
best_rmse = results_df.loc[1, "Mean RMSE"]
best_std = results_df.loc[1, "Std RMSE"]
print(f"Best model: {best_name}")
print(f"Mean 10-fold CV RMSE = {best_rmse:.4f}  (std = {best_std:.4f})")
print("\nRanking:")
for rank, row in results_df.iterrows():
    print(f"  {rank}. {row['Model']:<18} mean RMSE = {row['Mean RMSE']:.4f}")

## 15. Training the Final Model on the Full Dataset
The best pipeline is now fitted on **all** rows so the deployed model uses all available data.  
⚠️ The error measured on the same data it was trained on is *not* a fair test score. The honest performance estimate is the **cross-validation RMSE** from Section 14.

In [ ]:
final_model = clone(models[best_name])
final_model.fit(X, y)

train_pred = final_model.predict(X)
print("Training-set RMSE (NOT an independent test score):", round(float(np.sqrt(mean_squared_error(y, train_pred))), 4))
print("Cross-validation RMSE (the honest estimate)      :", round(float(best_rmse), 4))

## 16. Saving the Model and Results

In [ ]:
joblib.dump(final_model, "best_model.joblib")                       # full fitted pipeline
results_df.to_csv("model_results.csv", index_label="Rank")          # actual comparison results

# Extra info file used by the web app (feature order, input ranges, model name)
info = {
    "best_model": best_name,
    "cv_mean_rmse": float(best_rmse),
    "cv_std_rmse": float(best_std),
    "n_rows": int(len(df)),
    "features": FEATURES,
    "target": TARGET,
    "feature_ranges": {c: [float(X[c].min()), float(X[c].max())] for c in FEATURES},
    "sklearn_version": sklearn.__version__,
}
with open("best_model_info.json", "w") as f:
    json.dump(info, f, indent=2)

# Versions used in Colab -> use these in requirements.txt so the deployed app matches
with open("requirements_from_colab.txt", "w") as f:
    f.write(f"streamlit\nscikit-learn=={sklearn.__version__}\npandas=={pd.__version__}\nnumpy=={np.__version__}\njoblib=={joblib.__version__}\n")
print(open("requirements_from_colab.txt").read())

In [ ]:
# Verify the saved model loads and predicts (sample = first row of the dataset)
loaded = joblib.load("best_model.joblib")
sample = X.iloc[[0]]
print("Sample input:\n", sample.T)
print("Predicted price per unit area:", round(float(loaded.predict(sample)[0]), 3))
print("Actual price in dataset      :", float(y.iloc[0]))

In [ ]:
# Bundle the output files and download them (works in Colab)
import zipfile
out_files = ["best_model.joblib", "model_results.csv", "rmse_comparison.png",
             "best_model_info.json", "requirements_from_colab.txt"]
with zipfile.ZipFile("colab_outputs.zip", "w") as z:
    for f in out_files:
        z.write(f)
try:
    from google.colab import files
    files.download("colab_outputs.zip")
except ImportError:
    print("Not in Colab - files are saved in the current folder.")

## 17. Final Results and Conclusion
The next cell writes the conclusion from the **actual measured values**.

In [ ]:
print(f"All five algorithms (SVR, AdaBoost, Extra Trees, MLP Regressor, Gaussian Process) were evaluated on {len(df)} rows "
      f"with the same 10-fold cross-validation.")
print(f"The best model was {best_name} with a mean RMSE of {best_rmse:.4f} (std {best_std:.4f}) in price units of the target.")
worst = results_df.iloc[-1]
print(f"The weakest was {worst['Model']} with mean RMSE {worst['Mean RMSE']:.4f}.")
print("The best model was refit on the full dataset, saved as best_model.joblib, and is used by the Streamlit app.")
print("Limitations: small dataset from one region (Taiwan), only six features, no tuning beyond initial settings, "
      "and predictions are estimates, not guaranteed market prices.")